# Triaxial Compression — Reference State & Profile Evolution

Analysis for `triaxial_compression.lmp`.  Two parts:

1. **Reference state** (ε = 0, piston held at v = 0 over the 150k pre-compression window): averaged profiles **with confidence intervals** for the total stress σ_zzᵗ(z), the solvent-only stress σ_s,ss(z), the solvent mass density ρ_s(z), and the solvent mass fraction ρ_s/ρ_{s,0}(z).
2. **Profile evolution vs. timestep** for each of those observables, sampled as 10 curves starting **after** the slab reaches its 10 % compression (the relaxation phase).

Colorblind-friendly palette; gel interior shaded; in/out-gel means annotated.

## Before running: files to copy from the cluster

Copy the run's output from the cluster into your local `flow_data_local` tree.
Set `RUN_ID` and `sim_name` in the **Config** cell to match the run, then place
the files as below (the notebook builds every path from those two strings).

**Into** `flow_data_local/compression/<RUN_ID>/`  *(cluster:* `.../output_files/`*)*

| file pattern | cluster subfolder |
|---|---|
| `sigmazz_polymer_<sim_name>.dat`, `sigmazz_solvent_<sim_name>.dat` | `stress_data/` |
| `sigmazz_polymer_ref_<sim_name>.dat`, `sigmazz_solvent_ref_<sim_name>.dat` | `stress_data/` |
| `strain_zz_<sim_name>.dat` | `stress_data/` |
| `solvent_density_z_<sim_name>.dat`, `solvent_density_z_ref_<sim_name>.dat` | `chemical_potential/` |
| `piston_position_<sim_name>.dat` | `piston_data/` |
| `pairs_<sim_name>.dump`, `pairs_ref_<sim_name>.dump` | `pair_data/` |
| `polymer_pairs_<sim_name>.dump`, `polymer_pairs_ref_<sim_name>.dump` | `pair_data/` |
| `bonds_<sim_name>.dump`, `bonds_ref_<sim_name>.dump` | `pair_data/` |

**Into** `flow_data_local/traj_files.nosync/`  *(cluster:* `.../traj_files/`*)*

| file pattern | cluster subfolder |
|---|---|
| `traj_stress_<sim_name>.lammpstrj` | `traj_files/` |
| `traj_ref_<sim_name>.lammpstrj` | `traj_files/` |

Plots are written to `flow_data_local/plots/compression/<RUN_ID>/` (created automatically).

*Optional:* the run also writes high-resolution `sigmazz_*_fine_support/piston_*` and
`solvent_density_z_fine_*` files (support/piston windows). This notebook plots the
coarse profiles only; copy the `*_fine_*` files too if you later add high-res panels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from scipy import stats
from scipy.interpolate import interp1d
from pathlib import Path

# ---- user rcParams (matches compression_analysis.ipynb) ----
plt.rcParams.update({
    'font.family': 'CMU Serif',
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'CMU Serif',
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 25,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
    'legend.fontsize': 23,
    'figure.titlesize': 22,
    'axes.unicode_minus': False,
})

# ---- colorblind-friendly palettes ----
# Wong (2011) categorical palette for the reference single-curve plots.
WONG = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73','vermillion':'#D55E00',
        'skyblue':'#56B4E9','yellow':'#F0E442','reddishpurple':'#CC79A7','black':'#000000'}
# 'cividis' is the most CVD-safe sequential map -> used for the time gradient.
EVO_CMAP = 'cividis'
GEL_SHADE = dict(color='0.6', alpha=0.15, zorder=0)   # neutral grey, CVD-safe

print('Imports + style ready')

In [ ]:
# ── CONFIG: only change these lines to switch datasets ─────────────────────
RUN_ID   = "rho04_p1.52_600k_4M_1"   # folder inside flow_data_local/compression/
sim_name = "walled_slab_support_5beads_tall_rho04_p1.52_1.0_1.0_600000_1.0_1.0_4000000"
# ───────────────────────────────────────────────────────────────────────────

DATA_DIR  = Path("../../flow_data_local/compression") / RUN_ID
PLOT_DIR  = Path("../../flow_data_local/plots/compression") / RUN_ID
TRAJ_DIR  = Path("../../flow_data_local/traj_files.nosync")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ---- analysis parameters ----
binWidth      = 2.0      # coarse z-bin (sigma); must match triaxial_compression.lmp
binWidth_fine = 0.5      # fine z-bin (sigma)        "
# kinetic stress removed per request -- all recorded stresses are virial-only
solvent_mass  = 1.0      # solvent bead mass (LJ); rho_mass = solvent_mass * rho_number
comp_percent  = 0.1      # compression strain threshold (halt); evolution starts after this
n_curves      = 10       # target time-evolution curves (matches num_stress_curves)
ci_level      = 0.95
gel_thresh    = 0.05     # gel interior = bins where |sigma_p,zz| > gel_thresh*max
flat_tol      = 0.15     # final evolution curve "flat inside gel" if rel. spread < this
BOND_SIGN     = +1.0     # flip to -1 if sigma_p,pp comes out opposite the group sigma_p

# ---- file paths ----
def D(name):  return DATA_DIR / f'{name}_{sim_name}.dat'
def Ddump(name): return DATA_DIR / f'{name}_{sim_name}.dump'
def T(name):  return TRAJ_DIR / f'{name}_{sim_name}.lammpstrj'

# stress (coarse): sigma_zz polymer/solvent, production + reference
F_SZZ_P      = D('sigmazz_polymer');      F_SZZ_S      = D('sigmazz_solvent')
F_SZZ_P_REF  = D('sigmazz_polymer_ref');  F_SZZ_S_REF  = D('sigmazz_solvent_ref')
# solvent density (coarse), production + reference  (cols: ... density/number density/mass)
F_DENS       = DATA_DIR / f'solvent_density_z_{sim_name}.dat'
F_DENS_REF   = DATA_DIR / f'solvent_density_z_ref_{sim_name}.dat'
# pair/bond dumps for ss and pp stress
F_PAIRS      = Ddump('pairs');            F_PAIRS_REF      = Ddump('pairs_ref')
F_PPAIRS     = Ddump('polymer_pairs');    F_PPAIRS_REF     = Ddump('polymer_pairs_ref')
F_BONDS      = Ddump('bonds');            F_BONDS_REF      = Ddump('bonds_ref')
F_TRAJSTRESS = T('traj_stress');          F_TRAJREF        = T('traj_ref')
# strain + piston position (to find the compression-halt timestep)
F_STRAIN     = D('strain_zz')
F_PISTON_POS = DATA_DIR / f'piston_position_{sim_name}.dat'
print('Config set for', sim_name)

In [ ]:
# ============================ HELPER FUNCTIONS =============================
def read_print_file(filepath, col_names=None):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    if col_names is None: col_names = [f'col_{i}' for i in range(arr.shape[1])]
    return {name: arr[:, i] for i, name in enumerate(col_names)}

def read_ave_time_file(filepath):
    """fix ave/time mode vector -> list of (timestep, bin_idx, values)."""
    out = []
    with open(filepath) as f:
        lines = [l for l in f if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) == 2:
            ts, nrows = int(parts[0]), int(parts[1])
            vals = []
            for j in range(1, nrows + 1):
                if i + j < len(lines):
                    vp = lines[i + j].split()
                    if len(vp) == 2: vals.append(float(vp[1]))
            if vals: out.append((ts, np.arange(1, len(vals)+1), np.array(vals)))
            i += nrows + 1
        else:
            i += 1
    return out

def read_ave_chunk_file(filepath):
    """fix ave/chunk -> list of (timestep, array[rows, cols]).
    cols: [chunk_id, Coord1, Ncount, val1(, val2...)]."""
    snaps = []
    with open(filepath) as f:
        lines = [l for l in f if l.strip() and not l.startswith('#')]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) in (2, 3):
            try: ts, nch = int(parts[0]), int(parts[1])
            except ValueError:
                i += 1; continue
            rows = []
            for j in range(1, nch + 1):
                if i + j < len(lines): rows.append([float(v) for v in lines[i + j].split()])
            if rows: snaps.append((ts, np.array(rows)))
            i += nch + 1
        else:
            i += 1
    return snaps

def read_strain_file(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    # cols: step  L_initial  L_current  -> eps = (L0 - L)/L0
    ts = arr[:, 0].astype(int); L0 = arr[:, 1]; L = arr[:, 2]
    eps = (L0 - L) / L0
    return ts, eps

def mean_ci(stack, ci=0.95):
    """stack: (n_samples, n_bins) -> (mean, lo, hi) per bin via t-interval."""
    stack = np.asarray(stack, float)
    n = stack.shape[0]
    m = np.nanmean(stack, axis=0)
    if n < 2:
        return m, m, m
    se = stats.sem(stack, axis=0, nan_policy='omit')
    half = se * stats.t.ppf(0.5 + ci/2, df=n-1)
    return m, m - half, m + half

def read_pairs_local_dump(filepath):
    """dump local (pair/local) -> list (timestep, box, data[n,7])
    data cols: id1 id2 type1 type2 fx fy fz."""
    frames = []
    with open(filepath) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'ITEM: TIMESTEP':
            ts = int(lines[i+1]); n = int(lines[i+3])
            xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
            zb = list(map(float, lines[i+7].split()))
            box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
            s = i + 9
            rows = [[float(v) for v in lines[s+j].split()] for j in range(n) if s+j < len(lines)]
            frames.append((ts, box, np.array(rows) if rows else np.zeros((0, 7))))
            i = s + n
        else:
            i += 1
    return frames

def read_bonds_dump(filepath):
    """dump local (bond/local) -> list (timestep, box, data[n,5])
    data cols: batom1 batom2 btype force dist."""
    frames = []
    with open(filepath) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'ITEM: TIMESTEP':
            ts = int(lines[i+1]); n = int(lines[i+3])
            xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
            zb = list(map(float, lines[i+7].split()))
            box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
            s = i + 9
            rows = [[float(v) for v in lines[s+j].split()] for j in range(n) if s+j < len(lines)]
            frames.append((ts, box, np.array(rows) if rows else np.zeros((0, 5))))
            i = s + n
        else:
            i += 1
    return frames

def _stream_traj_positions(traj_file, target_ts, types_keep):
    """Return {ts: (box, {id:(x,y,z)})} for atoms whose type is in types_keep."""
    out = {}
    with open(traj_file) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if 'ITEM: TIMESTEP' in lines[i]:
            ts = int(lines[i+1]); n = int(lines[i+3])
            if ts in target_ts:
                xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
                zb = list(map(float, lines[i+7].split()))
                box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
                pos = {}
                for j in range(n):
                    p = lines[i+9+j].split()
                    if int(p[1]) in types_keep:
                        pos[int(p[0])] = (float(p[3]), float(p[4]), float(p[5]))
                out[ts] = (box, pos)
            i += 9 + n
        else:
            i += 1
    return out

print('Readers defined')

In [ ]:
# ===== solvent-only (ss) and polymer-only (pp) stress from pair/bond dumps =====
def _virial_zz_from_pairs(pair_data, pos, box, binWidth, type_filter):
    """sigma_zz(z) from pair/local forces, both atoms in type_filter (a set).
    Returns (z_bins, sigma_zz) -- virial only, no kinetic term."""
    lx = box['x'][1]-box['x'][0]; ly = box['y'][1]-box['y'][0]; lz = box['z'][1]-box['z'][0]
    zlo = box['z'][0]
    nb = int(round(lz / binWidth)); binvol = lx*ly*binWidth
    z_bins = zlo + (np.arange(nb)+0.5)*binWidth
    szz = np.zeros(nb)
    if len(pair_data):
        m = np.isin(pair_data[:,2].astype(int), list(type_filter)) & \
            np.isin(pair_data[:,3].astype(int), list(type_filter))
        sp = pair_data[m]
        if len(sp):
            id1 = sp[:,0].astype(int); id2 = sp[:,1].astype(int); fz = sp[:,6]
            keep = np.array([(a in pos and b in pos) for a,b in zip(id1,id2)])
            if keep.any():
                id1=id1[keep]; id2=id2[keep]; fz=fz[keep]
                p1 = np.array([pos[a] for a in id1]); p2 = np.array([pos[b] for b in id2])
                dz = p2[:,2]-p1[:,2]; dz -= lz*np.round(dz/lz)
                w = -(dz*fz)
                zmid = p1[:,2] + dz*0.5
                bi = ((zmid - zlo)/binWidth).astype(int)
                ok = (bi>=0)&(bi<nb)
                np.add.at(szz, bi[ok], w[ok])
    return z_bins, szz/binvol

def _bond_virial_zz(bond_data, pos, box, binWidth, bond_sign=1.0):
    """FENE bond sigma_zz(z): W_zz = bond_sign*(force/|r|)*dz^2, midpoint-binned."""
    lx = box['x'][1]-box['x'][0]; ly = box['y'][1]-box['y'][0]; lz = box['z'][1]-box['z'][0]
    zlo = box['z'][0]
    nb = int(round(lz / binWidth)); binvol = lx*ly*binWidth
    szz = np.zeros(nb)
    if len(bond_data):
        b1 = bond_data[:,0].astype(int); b2 = bond_data[:,1].astype(int); force = bond_data[:,3]
        keep = np.array([(a in pos and b in pos) for a,b in zip(b1,b2)])
        if keep.any():
            b1=b1[keep]; b2=b2[keep]; force=force[keep]
            p1=np.array([pos[a] for a in b1]); p2=np.array([pos[b] for b in b2])
            dr = p2-p1
            dr[:,0]-=lx*np.round(dr[:,0]/lx); dr[:,1]-=ly*np.round(dr[:,1]/ly); dr[:,2]-=lz*np.round(dr[:,2]/lz)
            r = np.sqrt((dr**2).sum(1)); r[r==0]=np.nan
            w = bond_sign*(force/r)*dr[:,2]**2
            zmid = p1[:,2] + dr[:,2]*0.5
            bi = ((zmid - zlo)/binWidth).astype(int); ok=(bi>=0)&(bi<nb)&np.isfinite(w)
            np.add.at(szz, bi[ok], w[ok])
    return szz/binvol

def compute_ss_stress(pairs_file, traj_file, binWidth, z_target):
    """Per-frame sigma_s,ss(z), virial only, on z_target. Returns (timesteps, stack[n,nz])."""
    pf = read_pairs_local_dump(pairs_file)
    if not pf: return [], np.zeros((0,len(z_target)))
    tgt = {ts for ts,_,_ in pf}
    traj = _stream_traj_positions(traj_file, tgt, {3})
    ts_out, stack = [], []
    for ts, box, pdata in pf:
        if ts not in traj: continue
        _, pos = traj[ts]
        zb, szz = _virial_zz_from_pairs(pdata, pos, box, binWidth, {3})
        stack.append(np.interp(z_target, zb, szz, left=np.nan, right=np.nan)); ts_out.append(ts)
    return ts_out, np.array(stack)

def compute_pp_stress(ppairs_file, bonds_file, traj_file, binWidth, z_target, bond_sign=1.0):
    """Per-frame sigma_p,pp(z) = pp pair virial + FENE bond virial, virial only, on z_target."""
    pf = read_pairs_local_dump(ppairs_file)
    bf = {ts:(box,data) for ts,box,data in read_bonds_dump(bonds_file)} if Path(bonds_file).exists() else {}
    if not pf: return [], np.zeros((0,len(z_target)))
    tgt = {ts for ts,_,_ in pf}
    traj = _stream_traj_positions(traj_file, tgt, {1,2})
    ts_out, stack = [], []
    for ts, box, pdata in pf:
        if ts not in traj: continue
        _, pos = traj[ts]
        zb, szz = _virial_zz_from_pairs(pdata, pos, box, binWidth, {1,2})
        if ts in bf:
            szz = szz + _bond_virial_zz(bf[ts][1], pos, box, binWidth, bond_sign)
        stack.append(np.interp(z_target, zb, szz, left=np.nan, right=np.nan)); ts_out.append(ts)
    return ts_out, np.array(stack)

print('ss / pp stress reconstruction defined')

## Load data: coordinates, gel bounds, reference & production profiles

In [ ]:
# ---- coarse sigma_zz (production) -> z grid + total/partial time series ----
szz_p = read_ave_time_file(F_SZZ_P)
szz_s = read_ave_time_file(F_SZZ_S)
n_prod = len(szz_p)
prod_ts   = np.array([szz_p[i][0] for i in range(n_prod)])
sig_p_zz  = [szz_p[i][2] for i in range(n_prod)]
sig_s_zz  = [szz_s[i][2] for i in range(n_prod)]
sig_t_zz  = [sig_p_zz[i] + sig_s_zz[i] for i in range(n_prod)]      # total sigma_zz
bins_z    = szz_p[0][1]
z_coords  = bins_z * binWidth - binWidth/2.0
print(f'coarse stress: {n_prod} production snapshots, {len(z_coords)} z-bins '
      f'[{z_coords.min():.1f}, {z_coords.max():.1f}]')

# ---- reference coarse sigma_zz (multi-snapshot -> mean + CI) ----
szz_p_ref = read_ave_time_file(F_SZZ_P_REF)
szz_s_ref = read_ave_time_file(F_SZZ_S_REF)
ref_p_stack = np.array([s[2] for s in szz_p_ref])
ref_s_stack = np.array([s[2] for s in szz_s_ref])
ref_t_stack = ref_p_stack + ref_s_stack
print(f'reference stress: {len(szz_p_ref)} snapshots')

# total sigma_zz reference mean/CI
sig_t_ref_m, sig_t_ref_lo, sig_t_ref_hi = mean_ci(ref_t_stack, ci_level)
# group-based polymer reference (for the pp sign-validation cell)
sig_p_ref_m = np.nanmean(ref_p_stack, axis=0)

# ---- gel bounds from reference polymer sigma_zz ----
_pm = np.abs(sig_p_ref_m); _pmax = float(_pm.max())
_gel = (_pm > gel_thresh*_pmax) if _pmax > 0 else np.zeros(len(z_coords), bool)
z_gel_lo = float(z_coords[_gel].min()) if _gel.any() else z_coords[0]
z_gel_hi = float(z_coords[_gel].max()) if _gel.any() else z_coords[-1]
in_gel  = (z_coords >= z_gel_lo) & (z_coords <= z_gel_hi)
print(f'gel interior: z in [{z_gel_lo:.1f}, {z_gel_hi:.1f}]  ({in_gel.sum()} bins)')

# ---- compression-halt timestep (evolution starts AFTER this) ----
strain_ts, strain_eps = read_strain_file(F_STRAIN)
_reached = np.where(strain_eps >= comp_percent)[0]
halt_ts = int(strain_ts[_reached[0]]) if len(_reached) else int(strain_ts[-1])
print(f'compression halt at step {halt_ts} (eps>={comp_percent}); '
      f'{"reached" if len(_reached) else "NOT reached - using last step"}')

In [ ]:
# ---- solvent density (production + reference): cols chunk,Coord1,Ncount,n_dens,m_dens ----
def _load_density(path):
    snaps = read_ave_chunk_file(path)
    ts = np.array([s[0] for s in snaps])
    z  = snaps[0][1][:, 1]
    nden = np.array([s[1][:, 3] for s in snaps])        # density/number
    mden = np.array([s[1][:, 4] for s in snaps])        # density/mass
    return ts, z, nden, mden

dens_ts, dens_z, dens_n, dens_m = _load_density(F_DENS)
rdens_ts, rdens_z, rdens_n, rdens_m = _load_density(F_DENS_REF)
print(f'density: {len(dens_ts)} production, {len(rdens_ts)} reference snapshots on {len(dens_z)} bins')

# reference mass density mean/CI
rho_ref_m, rho_ref_lo, rho_ref_hi = mean_ci(rdens_m, ci_level)
# bulk reference solvent mass density rho_{s,0} = mean over reservoir bins (high-density)
_rmax = float(np.nanmax(rho_ref_m)); _res = rho_ref_m >= 0.85*_rmax
rho_s0 = float(np.nanmean(rho_ref_m[_res]))
print(f'rho_s,0 (bulk reservoir reference mass density) = {rho_s0:.4f}')

# reference mass fraction stack = each ref snapshot / rho_s0
mf_ref_stack = rdens_m / rho_s0
mf_ref_m, mf_ref_lo, mf_ref_hi = mean_ci(mf_ref_stack, ci_level)

In [ ]:
# ---- solvent-only (ss) and polymer-only (pp) stress: reference + production ----
# Reference: multi-frame *_ref dumps -> mean + CI.  Production: filter post-halt.
ss_ref_m = ss_ref_lo = ss_ref_hi = None
ss_prod_ts = None; ss_prod_stack = None
if F_PAIRS_REF.exists() and F_TRAJREF.exists():
    _ts, _stk = compute_ss_stress(F_PAIRS_REF, F_TRAJREF, binWidth, z_coords)
    if len(_stk): ss_ref_m, ss_ref_lo, ss_ref_hi = mean_ci(_stk, ci_level)
    print(f'sigma_s,ss reference: {len(_ts)} frames')
else:
    print('NOTE: ss reference dumps missing -', F_PAIRS_REF.name, '/', F_TRAJREF.name)

if F_PAIRS.exists() and F_TRAJSTRESS.exists():
    ss_prod_ts, ss_prod_stack = compute_ss_stress(F_PAIRS, F_TRAJSTRESS, binWidth, z_coords)
    ss_prod_ts = np.array(ss_prod_ts)
    print(f'sigma_s,ss production: {len(ss_prod_ts)} frames')
else:
    print('NOTE: ss production dumps missing')

pp_ref_m = pp_ref_lo = pp_ref_hi = None
pp_prod_ts = None; pp_prod_stack = None
if F_PPAIRS_REF.exists() and F_TRAJREF.exists():
    _ts, _stk = compute_pp_stress(F_PPAIRS_REF, F_BONDS_REF, F_TRAJREF, binWidth, z_coords, BOND_SIGN)
    if len(_stk): pp_ref_m, pp_ref_lo, pp_ref_hi = mean_ci(_stk, ci_level)
    print(f'sigma_p,pp reference: {len(_ts)} frames')
if F_PPAIRS.exists() and F_TRAJSTRESS.exists():
    pp_prod_ts, pp_prod_stack = compute_pp_stress(F_PPAIRS, F_BONDS, F_TRAJSTRESS, binWidth, z_coords, BOND_SIGN)
    pp_prod_ts = np.array(pp_prod_ts)
    print(f'sigma_p,pp production: {len(pp_prod_ts)} frames')

In [ ]:
# ---- sigma_p,pp sign validation (group sigma_p_ref should track pp+kin) ----
# The group-based reference polymer stress (incl. bonds + 1/2 ps cross) should
# have the SAME sign and comparable magnitude as the pp reconstruction.  If the
# pp curve is mirrored about zero, set BOND_SIGN = -1 in the Config cell.
if pp_ref_m is not None:
    _ig = in_gel
    corr = np.corrcoef(pp_ref_m[_ig], sig_p_ref_m[_ig])[0, 1]
    print(f'corr(sigma_p,pp_ref, group sigma_p_ref) inside gel = {corr:+.3f}')
    if corr < 0:
        print('  >>> NEGATIVE correlation: sigma_p,pp likely sign-flipped. '
              'Set BOND_SIGN = -1.0 in Config and re-run.')
    else:
        print('  sign looks consistent (positive correlation).')
else:
    print('pp reference unavailable - skipping sign check.')

## Plotting helpers

In [ ]:
def _means_text(z, curve, in_gel):
    """Mean of `curve` inside and outside the gel as a formatted string."""
    mi = np.nanmean(curve[in_gel]) if in_gel.any() else np.nan
    mo = np.nanmean(curve[~in_gel]) if (~in_gel).any() else np.nan
    return mi, mo, f'mean in gel = {mi:.3g}\nmean out gel = {mo:.3g}'

def _final_flat_inside(curve, in_gel, tol):
    """True if `curve` is ~flat inside the gel (relative spread < tol)."""
    v = curve[in_gel]; v = v[np.isfinite(v)]
    if len(v) < 3: return False
    denom = max(abs(np.nanmean(v)), 1e-9)
    return (np.nanstd(v) / denom) < tol

def shade_gel(ax):
    ax.axvspan(z_gel_lo, z_gel_hi, **GEL_SHADE)

def plot_reference(ax, z, m, lo, hi, color, ylabel, title, annotate=True):
    ax.fill_between(z, lo, hi, color=color, alpha=0.25, lw=0, zorder=2)
    ax.plot(z, m, '-', color=color, lw=2.5, zorder=3)
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    shade_gel(ax)
    ax.set_xlabel(r'$z\ (\sigma)$'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(z.min(), z.max()); ax.grid(alpha=0.3)
    if annotate:
        mi, mo, txt = _means_text(z, m, in_gel)
        ax.text(0.02, 0.97, txt, transform=ax.transAxes, va='top', ha='left',
                fontsize=15, bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

def subsample(ts, stack, k):
    """Evenly pick up to k snapshots (keep order; always include the last)."""
    n = len(ts)
    if n <= k: idx = np.arange(n)
    else:      idx = np.unique(np.linspace(0, n-1, k).round().astype(int))
    return ts[idx], stack[idx]

def plot_evolution(ax, z, ts, stack, ylabel, title):
    """Time-coloured profiles (cividis); final curve bold black; gel shaded.
    Mean-in/out annotation ONLY if the final curve is flat inside the gel."""
    ts = np.asarray(ts); stack = np.asarray(stack)
    norm = Normalize(vmin=ts.min(), vmax=ts.max())
    cmap = plt.get_cmap(EVO_CMAP)
    for i in range(len(ts)):
        is_last = (i == len(ts)-1)
        ax.plot(z, stack[i], '-',
                color=('k' if is_last else cmap(norm(ts[i]))),
                lw=(3.5 if is_last else 1.6),
                alpha=(1.0 if is_last else 0.75),
                zorder=(5 if is_last else 3))
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    shade_gel(ax)
    ax.set_xlabel(r'$z\ (\sigma)$'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(z.min(), z.max()); ax.grid(alpha=0.3)
    sm = plt.cm.ScalarMappable(cmap=EVO_CMAP, norm=norm); sm.set_array([])
    cb = ax.figure.colorbar(sm, ax=ax, fraction=0.046, pad=0.02); cb.set_label('timestep')
    if _final_flat_inside(stack[-1], in_gel, flat_tol):
        mi, mo, txt = _means_text(z, stack[-1], in_gel)
        ax.text(0.02, 0.97, 'final (equilibrated)\n'+txt, transform=ax.transAxes,
                va='top', ha='left', fontsize=14,
                bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))
    else:
        ax.text(0.02, 0.97, 'final not flat inside\n(means omitted)', transform=ax.transAxes,
                va='top', ha='left', fontsize=13, color='0.35')

print('Plot helpers ready')

## 1 — Reference-state profiles (ε = 0) with confidence intervals

Averaged over the 150k pre-compression window (piston held at v = 0).  Bands are
the 95 % CIs across the reference snapshots; grey shading marks the gel interior.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle(f'Reference state (ε = 0):  {sim_name}', fontsize=14, fontweight='bold')

# (a) total sigma_zz = sigma_p,zz + sigma_s,zz
plot_reference(axes[0,0], z_coords, sig_t_ref_m, sig_t_ref_lo, sig_t_ref_hi,
               WONG['blue'], r'$\sigma_{zz}^{t}(z)$', r'(a) Total stress $\sigma_{zz}^{t}=\sigma_{p,zz}+\sigma_{s,zz}$')

# (b) solvent-only sigma_s,ss
if ss_ref_m is not None:
    plot_reference(axes[0,1], z_coords, ss_ref_m, ss_ref_lo, ss_ref_hi,
                   WONG['vermillion'], r'$\sigma_{s,zz}^{ss}(z)$', r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$')
else:
    axes[0,1].text(0.5,0.5,'ss reference\nunavailable',ha='center',va='center',transform=axes[0,1].transAxes)
    axes[0,1].set_title(r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$')

# (c) solvent mass density rho_s(z)
plot_reference(axes[1,0], dens_z, rho_ref_m, rho_ref_lo, rho_ref_hi,
               WONG['green'], r'$\rho_s(z)\ (m\,\sigma^{-3})$', r'(c) Solvent mass density $\rho_s$')
axes[1,0].axhline(rho_s0, color=WONG['black'], ls=':', lw=1.5, alpha=0.7)
axes[1,0].text(0.98, 0.05, r'$\rho_{s,0}$'+f' = {rho_s0:.3f}', transform=axes[1,0].transAxes,
               ha='right', va='bottom', fontsize=15)

# (d) mass fraction rho_s/rho_s0
plot_reference(axes[1,1], dens_z, mf_ref_m, mf_ref_lo, mf_ref_hi,
               WONG['reddishpurple'], r'$\rho_s/\rho_{s,0}$', r'(d) Solvent mass fraction $\rho_s/\rho_{s,0}$')
axes[1,1].axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)

out = PLOT_DIR / f'reference_state_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

## 2 — Profile evolution vs. timestep (relaxation phase)

Each observable sampled as up to 10 curves **starting after** the 10 % compression
halt (step shown above).  Colour = timestep (cividis); the **bold black** curve is
the final (most-relaxed) profile.  Gel interior shaded.  Per the request, no extra
time-averaged curve is drawn, and in/out-gel means are written only when the final
curve is flat inside the gel (i.e. has equilibrated).

In [ ]:
# Build post-halt evolution series for each observable, subsampled to ~n_curves.
# Total sigma_zz and density use their own snapshot grids; ss/pp use pair-dump grid.
def post_halt(ts, stack):
    ts = np.asarray(ts); stack = np.asarray(stack)
    m = ts >= halt_ts
    if m.sum() < 2:            # halt at/after last snapshot -> use last few frames
        m = np.zeros(len(ts), bool); m[-min(len(ts), n_curves):] = True
    return subsample(ts[m], stack[m], n_curves)

# total sigma_zz
tzz_ts, tzz_ev = post_halt(prod_ts, np.array(sig_t_zz))
# solvent mass density + mass fraction
dlm = dens_ts >= 0   # all
dmf_stack = dens_m / rho_s0
rho_ts, rho_ev = post_halt(dens_ts, dens_m)
mf_ts,  mf_ev  = post_halt(dens_ts, dmf_stack)

fig, axes = plt.subplots(2, 2, figsize=(17, 12), constrained_layout=True)
fig.suptitle(f'Profile evolution after {halt_ts}-step compression halt:  {sim_name}',
             fontsize=14, fontweight='bold')

plot_evolution(axes[0,0], z_coords, tzz_ts, tzz_ev,
               r'$\sigma_{zz}^{t}(z,t)$', r'(a) Total stress $\sigma_{zz}^{t}$')

if ss_prod_stack is not None and len(ss_prod_stack):
    ss_ts2, ss_ev = post_halt(ss_prod_ts, ss_prod_stack)
    plot_evolution(axes[0,1], z_coords, ss_ts2, ss_ev,
                   r'$\sigma_{s,zz}^{ss}(z,t)$', r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$')
else:
    axes[0,1].text(0.5,0.5,'ss production\nunavailable',ha='center',va='center',transform=axes[0,1].transAxes)

plot_evolution(axes[1,0], dens_z, rho_ts, rho_ev,
               r'$\rho_s(z,t)\ (m\,\sigma^{-3})$', r'(c) Solvent mass density $\rho_s$')

plot_evolution(axes[1,1], dens_z, mf_ts, mf_ev,
               r'$\rho_s/\rho_{s,0}$', r'(d) Solvent mass fraction $\rho_s/\rho_{s,0}$')
axes[1,1].axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)

out = PLOT_DIR / f'profile_evolution_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

### Optional — polymer-only stress σ_p,pp evolution

Provided separately since it is the new observable.  Same conventions as above.
If σ_p,pp looks sign-flipped relative to the group polymer stress (see the
validation cell), set `BOND_SIGN = -1.0` in Config and re-run.

In [ ]:
if pp_prod_stack is not None and len(pp_prod_stack):
    fig, axes = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    if pp_ref_m is not None:
        plot_reference(axes[0], z_coords, pp_ref_m, pp_ref_lo, pp_ref_hi,
                       WONG['orange'], r'$\sigma_{p,zz}^{pp}(z)$',
                       r'Reference $\sigma_{p,zz}^{pp}$ (pair + bond, virial only)')
    else:
        axes[0].text(0.5,0.5,'pp reference unavailable',ha='center',va='center',transform=axes[0].transAxes)
    pp_ts2, pp_ev = post_halt(pp_prod_ts, pp_prod_stack)
    plot_evolution(axes[1], z_coords, pp_ts2, pp_ev,
                   r'$\sigma_{p,zz}^{pp}(z,t)$', r'Evolution $\sigma_{p,zz}^{pp}$')
    out = PLOT_DIR / f'polymer_only_stress_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('sigma_p,pp production data unavailable - skipping.')